In [1]:
# https://www.kaggle.com/competitions/neoai-2026-day-2-audio

!pip install -q /kaggle/input/competitions/neoai-2026-day-2-audio/dataset/wheels/librosa-0.11.0-py3-none-any.whl \
                /kaggle/input/competitions/neoai-2026-day-2-audio/dataset/wheels/soundfile-0.13.1-py2.py3-none-any.whl \
                /kaggle/input/competitions/neoai-2026-day-2-audio/dataset/wheels/tiktoken-0.12.0-cp312-cp312-manylinux_2_28_x86_64.whl

In [2]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
from torch import nn
import torch.nn.functional as F

INPUT_DIR = Path('/kaggle/input/competitions/neoai-2026-day-2-audio/dataset')
sys.path.insert(0, str(INPUT_DIR))
import whisper_lib
print('whisper_lib from:', whisper_lib.__file__)

WHISPER_MODEL = 'large-v3-turbo'
BATCH_SIZE    = 64

whisper_lib from: /kaggle/input/competitions/neoai-2026-day-2-audio/dataset/whisper_lib.py


In [3]:
train_df = pd.read_csv(INPUT_DIR / 'train.csv')
test_df  = pd.read_csv(INPUT_DIR / 'test.csv')

# audio_path is relative to the dataset root; make it absolute for librosa.load
train_df['audio_path'] = train_df['audio_path'].apply(lambda p: str(INPUT_DIR / p))
test_df['audio_path']  = test_df['audio_path'].apply(lambda p: str(INPUT_DIR / p))

print(f'train: {len(train_df)} clips across {train_df.speaker_id.nunique()} speakers')
print(f'test : {len(test_df)} clips across {test_df.speaker_id.nunique()} speakers')
train_df.head()

train: 865 clips across 30 speakers
test : 280 clips across 30 speakers


,audio_id,speaker_id,audio_path
0,speaker_1088_train_000,speaker_1088,/kaggle/input/competitions/neoai-2026-day-2-au...
1,speaker_1088_train_001,speaker_1088,/kaggle/input/competitions/neoai-2026-day-2-au...
2,speaker_1088_train_002,speaker_1088,/kaggle/input/competitions/neoai-2026-day-2-au...
3,speaker_1088_train_003,speaker_1088,/kaggle/input/competitions/neoai-2026-day-2-au...
4,speaker_1088_train_004,speaker_1088,/kaggle/input/competitions/neoai-2026-day-2-au...


In [4]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

# Load an audio file
file_path = train_df['audio_path'][0]  # Replace with your audio file path
y, sample_rate = librosa.load(file_path, sr=None)
speed_factor = 1.5
y_stretched = librosa.effects.time_stretch(y, rate=speed_factor)

In [5]:
Audio(data=y, rate=sample_rate)

In [6]:
Audio(data=y_stretched, rate=sample_rate)

In [7]:
Audio(test_df['audio_path'][4])

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [9]:
# model = whisper_lib.load_model(WHISPER_MODEL).encoder.to(device)
# model.eval()

In [10]:
# def get_embeddings(paths, batch_size=BATCH_SIZE, speeds=[1.0, 1.5, 2.0,]):
#     inputs = []
#     for path in paths:
#         y, _ = librosa.load(file_path, sr=None)
#         for speed in speeds:
#             y_stretched = librosa.effects.time_stretch(y, rate=speed)
#             inputs.append(whisper_lib.pad_or_trim(torch.tensor(y_stretched).unsqueeze(0)))
#     inputs = torch.cat(inputs)

#     all_embs = []
    
#     for i in tqdm(range(0, len(inputs), batch_size)):
#         portion = inputs[i:i+batch_size]
#         mel = whisper_lib.log_mel_spectrogram(portion, 128).to(device)
#         with torch.no_grad():
#             embs = model(mel).detach().cpu()
#         all_embs.append(embs)

#     all_embs = torch.cat(all_embs)
#     return all_embs

In [11]:
# train_embs = get_embeddings(train_df['audio_path'])

In [ ]:
# def get_mels(paths, batch_size=BATCH_SIZE, speeds=[1.0, 1.5, 2.0, 2.5]):
#     inputs = []
#     for path in paths:
#         y, _ = librosa.load(file_path, sr=None)
#         for speed in speeds:
#             y_stretched = librosa.effects.time_stretch(y, rate=speed)
#             inputs.append(whisper_lib.pad_or_trim(torch.tensor(y_stretched).unsqueeze(0)))
#     inputs = torch.cat(inputs)

#     all_mels = []
    
#     for i in tqdm(range(0, len(inputs), batch_size)):
#         portion = inputs[i:i+batch_size]
#         mel = whisper_lib.log_mel_spectrogram(portion, 128)
#         all_mels.append(mel)

#     all_mels = torch.cat(all_mels)
#     return all_mels

In [17]:
import os
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Function to extract features from audio files
def extract_features(file_path, speeds=[1.0, 1.5, 2.0]):
    # Load the audio file
    y, sr = librosa.load(file_path, sr=None)

    features = []
    
    for speed in speeds:
        y_stretched = librosa.effects.time_stretch(y, rate=speed)
        
        features.append({
            'mfcc': np.mean(librosa.feature.mfcc(y=y_stretched, sr=sr, n_mfcc=13).T, axis=0),
            'chroma': np.mean(librosa.feature.chroma_stft(y=y_stretched, sr=sr).T, axis=0),
            'mel': np.mean(librosa.feature.melspectrogram(y=y_stretched, sr=sr).T, axis=0),
            'contrast': np.mean(librosa.feature.spectral_contrast(y=y_stretched, sr=sr).T, axis=0),
            'tonnetz': np.mean(librosa.feature.tonnetz(y=y_stretched, sr=sr).T, axis=0)
        })
    
    # Combine all features into a single array
    return np.stack([np.hstack(list(f.values())) for f in features])

In [43]:
X_train = []

for i in tqdm(range(len(train_df))):
    X_train.append(extract_features(train_df['audio_path'][i]))

X_train = np.concatenate(X_train)
X_train.shape

  0%|          | 0/865 [00:00<?, ?it/s]

(2595, 166)

In [58]:
y_train = np.array([0, 1, 1] * len(train_df))
y_train.shape

(2595,)

In [59]:
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(X_train, y_train, stratify=y_train, random_state=42, test_size=0.1)
X_tr.shape, X_va.shape, y_tr.shape, y_va.shape

((2335, 166), (260, 166), (2335,), (260,))

In [60]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.fit(X_tr, y_tr)

RandomForestClassifier()

In [61]:
from sklearn.metrics import roc_auc_score

score = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
score

np.float64(0.993821008570859)

In [48]:
X_test = []

for i in tqdm(range(len(test_df))):
    X_test.append(extract_features(test_df['audio_path'][i], speeds=[1.0]))

X_test = np.concatenate(X_test)
X_test.shape

  0%|          | 0/280 [00:00<?, ?it/s]

(280, 166)

In [62]:
submission = pd.DataFrame({
    'audio_id': test_df['audio_id'],
    'score':    model.predict_proba(X_test)[:, 1]
})
submission.to_csv('submission.csv', index=False)
print(f'wrote submission.csv ({len(submission)} rows)')
submission.head()

wrote submission.csv (280 rows)


,audio_id,score
0,speaker_1088_test_000,0.96
1,speaker_1088_test_002,0.50
2,speaker_1088_test_003,0.97
3,speaker_1088_test_004,0.69
4,speaker_1088_test_005,1.00
